# Spectrum prototype (Person C)

Each SPHEREx frame sees a star at **one** wavelength (linear variable filter), so photometry on every frame covering a target = its 0.75–5 µm spectrum. This notebook is the scratchpad behind `api/science/spectrum.py`.

Run from repo root with `BLINK_CACHE=cache`. First run downloads ~250 cutouts from IRSA (10–20 min); later runs use the cache.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('BLINK_CACHE', os.path.abspath('../cache'))
import numpy as np
import matplotlib.pyplot as plt
from api.science.spectrum import build_spectrum, load_targets
from api.science.features import interpret

## 1. Build the spectrum for one target

In [ ]:
tid = 'wise-0855'
t = load_targets()[tid]
r = build_spectrum(t['ra_2026'], t['dec_2026'], tid)
print(r['n_frames'], 'frames,', len(r['points']), 'usable points,', r['n_failed'], 'failed downloads')
print('blends:', r['blends'])

## 2. Plot raw points, binned spectrum and the bands the detector calls *seen*

In [ ]:
pts, b = r['points'], r['binned']
fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar([p['wavelength_um'] for p in pts], [p['flux'] for p in pts], [p['err'] for p in pts], fmt='.', alpha=.25, label='frames')
ax.errorbar([p['wavelength_um'] for p in b], [p['flux'] for p in b], [p['err'] for p in b], fmt='o-', label='binned R=40')
colors = {'seen': 'tab:green', 'not_clear': '0.85', 'opposite': 'tab:red', 'no_coverage': 'white'}
for band in r['features']['bands']:
    ax.axvspan(band['lo'], band['hi'], color=colors[band['status']], alpha=.3)
    ax.text(band['center_um'], ax.get_ylim()[1]*.95, band['molecule'], ha='center', fontsize=8)
ax.set(xlabel='wavelength (µm)', ylabel='flux (mJy)', title=t['name']); ax.legend()

## 3. Band table

In [ ]:
for band in r['features']['bands']:
    print(f"{band['id']:12s} {band['status']:12s} contrast={band['contrast']}  snr={band['snr']}")

## Notes from prototyping

- Aperture r=2 px (12″), sky annulus 4–7 px; flagged pixel on the star → frame skipped (no interpolation).
- Proper motion applied per frame from `data/targets.json` (Barnard's Star moves ~1.7 px per year).
- Errors: VARIANCE summed in aperture + sky-median error; binned errors inflated by √χ² when points scatter.
- **WISE 0855:** real signal only at ≥4 µm (~0.6 mJy, matches WISE W2≈13.9). Below 3 µm the light is a G=16.3 Gaia star 7″ away — reported in `blends`. 6″ pixels can't split them.